# LangSmith setup

LangSmith is LangChain's tracing / observability tool: once enabled, every `Runnable` call in this Python process (retrievers, `ChatHuggingFace`, chains built with `|`, everything) gets logged to the LangSmith UI automatically — no code changes to the chains themselves. You get to see, for every run: the full input/output, how long each step took, what got retrieved, the exact prompt sent to the LLM, and errors with a stack trace, all nested in one tree per `.invoke()` call.

It works the same way regardless of which LLM/embedding provider you're calling — OpenAI, Hugging Face local pipelines, anything wrapped as a LangChain `Runnable` — because the tracing hooks live in `langchain-core`, not in any specific provider integration.

**One thing worth being explicit about:** tracing is just environment variables (`LANGSMITH_TRACING`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT`) read by `langchain-core` at call time. Each Jupyter notebook is its own separate kernel/process, so running *this* notebook does not turn on tracing inside your other already-open notebooks. What actually makes tracing "stick" project-wide is the `.env` file created below — add the two-line snippet from the last cell to the top of any other notebook and it'll pick up the same key/project.

`LANGSMITH_TRACING=true` is all `langchain-core` needs to start attaching a tracing callback to every `Runnable.invoke()` in this process. Running any chain — even a trivial one — should produce a run in the LangSmith UI within a few seconds.

## 0. Prerequisite: get a free API key (one-time, outside this notebook)

1. Go to https://smith.langchain.com and sign up (GitHub/Google/email, no card needed) — this puts you on the free Developer plan with a monthly free trace quota.
2. Click your workspace name (top left) → **Settings** → **API Keys** → **Create API Key**.
3. Copy the key (starts with `lsv2_...`) — you only see it once.


```
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=lsv2_your_real_key_here
LANGSMITH_PROJECT=rag-from-scratch
LANGSMITH_ENDPOINT=https://api.smith.langchain.com
```

`.env` is already in `.gitignore`, so the key never gets committed.

In [3]:
import os
from dotenv import load_dotenv

# reads project-root .env into this process's environment
load_dotenv()  

required = ["LANGSMITH_API_KEY", "LANGSMITH_TRACING", "LANGSMITH_PROJECT"]
missing = [v for v in required if not os.environ.get(v)]
if missing:
    raise RuntimeError(
        f"Missing from .env: {missing}. Copy .env.example to .env and fill in your "
        "LangSmith API key (see the cell above for how to get one)."
    )

print("LangSmith env vars loaded:")
print(f"  LANGSMITH_TRACING = {os.environ['LANGSMITH_TRACING']}")
print(f"  LANGSMITH_PROJECT = {os.environ['LANGSMITH_PROJECT']}")
print(f"  LANGSMITH_ENDPOINT = {os.environ.get('LANGSMITH_ENDPOINT', 'https://api.smith.langchain.com')}")
print(f"  LANGSMITH_API_KEY  = {os.environ['LANGSMITH_API_KEY'][:8]}... (hidden)")

LangSmith env vars loaded:
  LANGSMITH_TRACING = true
  LANGSMITH_PROJECT = rag-from-scratch
  LANGSMITH_ENDPOINT = https://api.smith.langchain.com
  LANGSMITH_API_KEY  = lsv2_pt_... (hidden)


## 2. Confirm it landed, without leaving the notebook

Instead of just eyeballing the UI, query the LangSmith API directly for the run we just created — if this finds it, tracing is fully working end to end.

In [4]:
import time
from langsmith import Client

client = Client()

time.sleep(2)  # tracing is sent asynchronously; give it a moment to arrive

runs = list(
    client.list_runs(
        project_name=os.environ["LANGSMITH_PROJECT"],
        filter='eq(name, "langsmith_setup_smoke_test")',
        limit=1,
    )
)

if runs:
    run = runs[0]
    print("Found the traced run in LangSmith:")
    print(f"  name   : {run.name}")
    print(f"  run_id : {run.id}")
    print(f"  status : {run.status}")
    print("\nOpen https://smith.langchain.com , select the "
          f"'{os.environ['LANGSMITH_PROJECT']}' project, and you'll see it in the runs table.")
else:
    print("Not found yet — wait a few more seconds and re-run this cell, "
          "or check the project name matches LANGSMITH_PROJECT.")

C:\Users\Amine\AppData\Local\Temp\ipykernel_37684\3492527086.py:9: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#runs-query for the migration guide.
  client.list_runs(


Found the traced run in LangSmith:
  name   : langsmith_setup_smoke_test
  run_id : 01a014c1-3d73-79c3-a0c1-226c9f22c589
  status : success

Open https://smith.langchain.com , select the 'rag-from-scratch' project, and you'll see it in the runs table.


## 3. Using this in the other notebooks

Paste these two lines at the top of any other notebook in this repo (e.g. `1_indexing_retrieval_generation.ipynb`) and every LangChain/`langchain-huggingface` call made in that notebook — embeddings, retriever, `ChatHuggingFace`, the full `rag_chain.invoke(...)` — will show up in LangSmith automatically:

```python
from dotenv import load_dotenv
load_dotenv()
```

Optional: give each notebook its own project name, so their traces don't all pile into one bucket — set this *after* `load_dotenv()`, since it overrides the value loaded from `.env`:

```python
import os
os.environ["LANGSMITH_PROJECT"] = "rag-from-scratch-indexing"
```

No other code changes needed — not to the chains, not to how models are constructed. That's the whole point of tracing living in `langchain-core`'s callback system rather than in each provider integration.